# Statistical Comparison: Predictive Coding vs Backpropagation on MNIST

Runs multiple independent training trials for both PC and backprop,
then performs statistical analysis to compare test accuracies.

**Architecture** (identical topology, different activations):

```
PC:       pixels(784) ──→ hidden1(256,sigmoid) ──→ hidden2(64,sigmoid) ──→ class(10,softmax)
Backprop: pixels(784) ──→ hidden1(256,relu)    ──→ hidden2(64,relu)    ──→ class(10,softmax)
```

**Reports:**
- Per-trial accuracy results table
- Mean ± standard error for each method
- Paired t-test with p-value
- Cohen's d effect size
- Power analysis: estimated n_trials needed for significance

## Imports & Setup

In [1]:
import jax
import optax

from fabricpc.nodes import Linear
from fabricpc.core.topology import Edge
from fabricpc.graph_assembly import TaskMap, graph
from fabricpc.graph_initialization import initialize_params
from fabricpc.core.activations import (
    SigmoidActivation,
    SoftmaxActivation,
    ReLUActivation,
)
from fabricpc.core.energy import CrossEntropyEnergy
from fabricpc.core.inference import InferenceSGD
from fabricpc.training import train_pcn, evaluate_pcn
from fabricpc.training.train_backprop import train_backprop, evaluate_backprop
from fabricpc.experiments import ExperimentArm, ABExperiment
from fabricpc.utils.data.dataloader import MnistLoader
from fabricpc import setup_jax

setup_jax()  # "cpu", "cuda" or "tpu"
jax.config.update("jax_default_prng_impl", "threefry2x32")

train_config = {"num_epochs": 20}
batch_size = 200
optimizer = optax.adamw(0.001, weight_decay=0.001)

/home/shamir/jax-cuda-venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Model Factories

In [2]:
def create_pc_model(rng_key):
    """Create PC model with sigmoid activations."""
    pixels = Linear(shape=(784,), name="pixels")
    hidden1 = Linear(shape=(256,), activation=SigmoidActivation(), name="hidden1")
    hidden2 = Linear(shape=(64,), activation=SigmoidActivation(), name="hidden2")
    output = Linear(
        shape=(10,),
        activation=SoftmaxActivation(),
        energy=CrossEntropyEnergy(),
        name="class",
    )
    structure = graph(
        nodes=[pixels, hidden1, hidden2, output],
        edges=[
            Edge(source=pixels, target=hidden1.slot("in")),
            Edge(source=hidden1, target=hidden2.slot("in")),
            Edge(source=hidden2, target=output.slot("in")),
        ],
        task_map=TaskMap(x=pixels, y=output),
        inference=InferenceSGD(eta_infer=0.05, infer_steps=20),
    )
    params = initialize_params(structure, rng_key)
    return params, structure


def create_backprop_model(rng_key):
    """Create backprop model with ReLU activations (to avoid vanishing gradients)."""
    pixels = Linear(shape=(784,), name="pixels")
    hidden1 = Linear(shape=(256,), activation=ReLUActivation(), name="hidden1")
    hidden2 = Linear(shape=(64,), activation=ReLUActivation(), name="hidden2")
    output = Linear(
        shape=(10,),
        activation=SoftmaxActivation(),
        energy=CrossEntropyEnergy(),
        name="class",
    )
    structure = graph(
        nodes=[pixels, hidden1, hidden2, output],
        edges=[
            Edge(source=pixels, target=hidden1.slot("in")),
            Edge(source=hidden1, target=hidden2.slot("in")),
            Edge(source=hidden2, target=output.slot("in")),
        ],
        task_map=TaskMap(x=pixels, y=output),
        inference=InferenceSGD(eta_infer=0.05, infer_steps=20),
    )
    params = initialize_params(structure, rng_key)
    return params, structure

## Run Experiment

In [3]:
n_trials = 10   # Number of independent training trials per method
verbose = False  # Set to True to show per-epoch output

print("=" * 70)
print("Statistical Comparison: Predictive Coding vs Backpropagation")
print("=" * 70)
print("Dataset: MNIST")
print("Architecture: 784 -> 256 -> 64 -> 10")
print("PC activations: sigmoid | Backprop activations: relu")
print(f"Epochs per trial: {train_config['num_epochs']}")
print(f"Number of trials: {n_trials}")
print()

arm_pc = ExperimentArm(
    name="PC",
    model_factory=create_pc_model,
    train_fn=train_pcn,
    eval_fn=evaluate_pcn,
    optimizer=optimizer,
    train_config=train_config,
)

arm_bp = ExperimentArm(
    name="Backprop",
    model_factory=create_backprop_model,
    train_fn=train_backprop,
    eval_fn=evaluate_backprop,
    optimizer=optimizer,
    train_config=train_config,
)

experiment = ABExperiment(
    arm_a=arm_pc,
    arm_b=arm_bp,
    metric="accuracy",
    data_loader_factory=lambda seed: (
        MnistLoader(
            "train",
            batch_size=batch_size,
            tensor_format="flat",
            shuffle=True,
            seed=seed,
        ),
        MnistLoader(
            "test",
            batch_size=batch_size,
            tensor_format="flat",
            shuffle=False,
        ),
    ),
    n_trials=n_trials,
    verbose=verbose,
)

results = experiment.run()
results.print_summary()

Statistical Comparison: Predictive Coding vs Backpropagation
Dataset: MNIST
Architecture: 784 -> 256 -> 64 -> 10
PC activations: sigmoid | Backprop activations: relu
Epochs per trial: 20
Number of trials: 10

--- Trial 1/10 (seed=0) ---


Epoch 20/20: 100%|██████████| 6000/6000 [00:56<00:00, 106.39it/s, energy=0.0008, epoch=20/20]


  PC: accuracy=0.9834  (train: 56.5s)
  Backprop: accuracy=0.9775  (train: 42.6s)
--- Trial 2/10 (seed=1000) ---


Epoch 20/20: 100%|██████████| 6000/6000 [00:57<00:00, 103.55it/s, energy=0.0005, epoch=20/20]


  PC: accuracy=0.9810  (train: 58.0s)
  Backprop: accuracy=0.9774  (train: 45.1s)
--- Trial 3/10 (seed=2000) ---


...

  PC: accuracy=0.9820  (train: 60.6s)
  Backprop: accuracy=0.9779  (train: 44.9s)
--- Trial 9/10 (seed=8000) ---


Epoch 20/20: 100%|██████████| 6000/6000 [00:59<00:00, 100.02it/s, energy=0.0011, epoch=20/20]


  PC: accuracy=0.9823  (train: 60.0s)
  Backprop: accuracy=0.9785  (train: 45.6s)
--- Trial 10/10 (seed=9000) ---


Epoch 20/20: 100%|██████████| 6000/6000 [00:56<00:00, 106.31it/s, energy=0.0007, epoch=20/20]


  PC: accuracy=0.9799  (train: 56.5s)
  Backprop: accuracy=0.9802  (train: 44.1s)
A/B Experiment: PC vs Backprop

--- Training Time per Epoch ---
PC: 2.909 +/- 0.029s
Backprop: 2.253 +/- 0.023s
Ratio: PC is 1.29x Backprop time

Total wall time: 1056.8s
Metric: accuracy
Trials: 10
Epochs per trial: 20
Design: Paired (same seed per trial)

Trial    Seed     PC%                  Backprop%            Diff%       
------------------------------------------------------------------------
1        0        98.34                97.75                +0.59       
2        1000     98.10                97.74                +0.36       
3        2000     98.01                98.13                -0.12       
4        3000     98.04                98.11                -0.07       
5        4000     97.93                97.95                -0.02       
6        5000     98.23                98.03                +0.20       
7        6000     98.10                97.74                +0.36       
8  